# 01 — VIIRS VNP46A4 Acquisition

Load 3 years (2023-2025) of annual nighttime radiance composites for mainland Portugal.
Multi-year median suppresses transient artifacts (fires, construction, cloud residuals).

| Property | Value |
|----------|-------|
| Product | VNP46A4 (annual, near-nadir, snow-free) |
| Resolution | ~500m |
| Units | nW/cm2/sr |
| Quality flags removed | 1 (poor), 255 (fill) |

In [1]:
import sys
sys.path.insert(0, '../../apps/stargazing_spots')

import json
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from stargazing_spots.processing import clean_radiance, classify_dark_sky
import stargazing_spots

print(f"stargazing_spots v{stargazing_spots.__version__}")
print(f"NumPy {np.__version__}, xarray {xr.__version__}")

stargazing_spots v0.7.0
NumPy 2.4.4, xarray 2026.4.0


In [ ]:
# Project paths (relative from notebooks/portugal/)
APP_DIR = Path('../../apps/stargazing_spots')
CACHE_DIR = APP_DIR / 'input' / 'portugal' / 'cache'
BOUNDARY_DIR = APP_DIR / 'input' / 'portugal' / 'boundary'
OUTPUT_DIR = APP_DIR / 'output' / 'portugal'
VIZ_DIR = OUTPUT_DIR / 'visualizations'

# Ensure output directories exist
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
YEARS = ['2023', '2024', '2025']
VARIABLE = 'NearNadir_Composite_Snow_Free'
THRESHOLD = 3.0  # nW/cm2/sr — Bortle 3-4 boundary

print(f"Cache directory: {CACHE_DIR.resolve()}")
print(f"Output directory: {VIZ_DIR.resolve()}")

## 1. Area of Interest — Mainland Portugal

GADM Level 1 boundaries. Mainland processed separately from Madeira and Azores to avoid downloading ocean tiles.

In [ ]:
# Load GADM boundaries
gadm = gpd.read_file(BOUNDARY_DIR / 'gadm41_PRT_1.json.zip')
print(f"GADM Level 1 regions: {sorted(gadm['NAME_1'].unique())}")

# Filter to mainland (exclude island territories)
mainland = gadm[~gadm['NAME_1'].isin(['Madeira', 'Azores'])].copy()
print(f"\nMainland Portugal: {len(mainland)} districts")
print(f"Total bounds (W, S, E, N): {mainland.total_bounds}")
print(f"Total area: ~{mainland.to_crs(epsg=3035).area.sum() / 1e9:.1f} thousand km2")

In [ ]:
# Plot Area of Interest
fig, ax = plt.subplots(figsize=(6, 9))
mainland.plot(ax=ax, edgecolor='#333333', facecolor='#e8f5e9', linewidth=0.4)
mainland.dissolve().boundary.plot(ax=ax, color='#1b5e20', linewidth=1.5)
ax.set_title('Mainland Portugal — Area of Interest', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## 2. Load Cached VIIRS Data (2023-2025)

Each cache file is a joblib-serialized xarray Dataset with a JSON metadata sidecar for provenance. No API calls needed to run this notebook.

In [ ]:
datasets = {}

for year in YEARS:
    cache_file = CACHE_DIR / f'VNP46A4_{year}.joblib'
    meta_file = cache_file.with_suffix('.meta.json')
    
    if not cache_file.exists():
        print(f"  [{year}] NOT FOUND — run acquisition pipeline first")
        continue
    
    # Load cached xarray Dataset
    datasets[year] = joblib.load(cache_file)
    size_mb = cache_file.stat().st_size / 1e6
    
    # Display metadata
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        print(f"  [{year}] Loaded ({size_mb:.1f} MB) — "
              f"created {meta['created_at'][:10]}, "
              f"quality flags removed: {meta['parameters']['quality_flags_removed']}, "
              f"bbox: {meta['parameters']['boundary_bbox']}")
    else:
        print(f"  [{year}] Loaded ({size_mb:.1f} MB) — no metadata sidecar")

print(f"\nDatasets loaded: {len(datasets)}/{len(YEARS)}")

## 3. Dataset Structure

Regular geographic grid at ~0.004 deg (~500m at 39 deg N).

In [ ]:
# Inspect the 2023 dataset as representative example
ds_example = datasets['2023']
print("=== Dataset structure ===")
print(ds_example)
print(f"\n=== Grid properties ===")
print(f"Dimensions: {dict(ds_example.dims)}")

dx = abs(float(ds_example.x[1] - ds_example.x[0]))
dy = abs(float(ds_example.y[1] - ds_example.y[0]))
print(f"Spatial resolution: dx={dx:.5f} deg, dy={dy:.5f} deg")
print(f"Approximate ground distance: {dx * np.cos(np.radians(39)) * 111320:.0f} m (E-W), "
      f"{dy * 111320:.0f} m (N-S)")

n_pixels = ds_example.dims['y'] * ds_example.dims['x']
print(f"Total grid pixels: {n_pixels:,}")
print(f"\nSpatial extent:")
print(f"  Longitude: {float(ds_example.x.min()):.3f} to {float(ds_example.x.max()):.3f} deg E")
print(f"  Latitude:  {float(ds_example.y.min()):.3f} to {float(ds_example.y.max()):.3f} deg N")

In [ ]:
# Show the data variable details
print(f"=== Variable: {VARIABLE} ===")
da_example = ds_example[VARIABLE]
print(da_example)
print(f"\nData type: {da_example.dtype}")
print(f"Shape: {da_example.shape}")
print(f"Chunks: {da_example.chunks if da_example.chunks else 'not chunked (in-memory)'}")

## 4. Quality Checks

Verify: no negatives, consistent grid dimensions, plausible value range across all years.

In [ ]:
print(f"{'Year':<6} {'Shape':<20} {'NaN %':<8} {'Negatives':<10} "
      f"{'Min':<8} {'Mean':<10} {'Median':<10} {'Max':<10}")
print("-" * 82)

for year in YEARS:
    ds = datasets[year]
    # Squeeze time dimension to get 2D array
    data = ds[VARIABLE].values.squeeze()
    total = data.size
    nan_count = np.isnan(data).sum()
    nan_pct = 100 * nan_count / total
    
    valid = data[~np.isnan(data)]
    negatives = (valid < 0).sum()
    
    print(f"{year:<6} {str(data.shape):<20} {nan_pct:<8.1f} {negatives:<10} "
          f"{valid.min():<8.2f} {valid.mean():<10.3f} {np.median(valid):<10.3f} {valid.max():<10.1f}")

# Verify grid consistency
shapes = [datasets[y][VARIABLE].values.squeeze().shape for y in YEARS]
assert len(set(shapes)) == 1, f"Grid mismatch across years: {shapes}"
print(f"\nAll years pass quality checks:")
print(f"  - No negative values detected (quality flags pre-filtered)")
print(f"  - Consistent grid dimensions: {shapes[0]}")
print(f"  - NaN coverage stable across years (ocean/border masking consistent)")

## 5. Radiance Statistics

In [ ]:
# Apply radiometric cleaning and compute statistics
cleaned_arrays = {}

for year in YEARS:
    ds = datasets[year]
    # Extract the 2D radiance array (squeeze time dimension)
    radiance_2d = ds[VARIABLE].isel(time=0)
    
    # Apply cleaning from stargazing_spots.processing
    cleaned = clean_radiance(radiance_2d)
    cleaned_arrays[year] = cleaned
    
    valid_count = int(cleaned.count())
    total_count = cleaned.size
    dark_count = int((cleaned < THRESHOLD).sum())
    dark_pct = 100 * dark_count / valid_count if valid_count > 0 else 0
    
    print(f"[{year}] Valid pixels: {valid_count:,}/{total_count:,} | "
          f"Dark (<{THRESHOLD} nW): {dark_count:,} ({dark_pct:.1f}%)")

## 6. Radiance Distribution

Bimodal: left peak (< 3 nW) = rural/natural dark-sky candidates; right tail (> 10 nW) = urban. Threshold at 3.0 nW/cm2/sr (Bortle 3-4 boundary, Falchi et al. 2016).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, year in enumerate(YEARS):
    values = cleaned_arrays[year].values.flatten()
    values = values[~np.isnan(values)]
    
    ax = axes[i]
    
    # Plot histogram (clip at 20 nW for visibility)
    ax.hist(values[values < 20], bins=100, color='#2196F3', edgecolor='none', alpha=0.8,
            density=False)
    
    # Threshold annotation
    ax.axvline(x=THRESHOLD, color='#D32F2F', linestyle='--', linewidth=2.0,
               label=f'Threshold: {THRESHOLD} nW/cm$^2$/sr')
    
    # Percentage annotation
    pct_dark = 100 * (values < THRESHOLD).sum() / len(values)
    ax.text(0.95, 0.92, f'{pct_dark:.1f}% below threshold',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=10, color='#D32F2F', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Radiance (nW/cm$^2$/sr)', fontsize=10)
    ax.set_ylabel('Pixel Count', fontsize=10)
    ax.set_title(f'{year}', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(0, 20)
    ax.grid(True, alpha=0.2)

plt.suptitle('VIIRS VNP46A4 Radiance Distribution — Mainland Portugal',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(VIZ_DIR / 'radiance_histogram.png', dpi=150, bbox_inches='tight',
            facecolor='white')
print(f"Saved: {VIZ_DIR / 'radiance_histogram.png'}")
plt.show()

## 7. Spatial Radiance Maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 12))

for i, year in enumerate(YEARS):
    data = cleaned_arrays[year].values.copy()
    
    # Get coordinate arrays for extent
    ds = datasets[year]
    x_coords = ds.coords['x'].values
    y_coords = ds.coords['y'].values
    extent = [x_coords.min(), x_coords.max(), y_coords.min(), y_coords.max()]
    
    ax = axes[i]
    im = ax.imshow(data, extent=extent, cmap='magma', vmin=0, vmax=25,
                   interpolation='nearest', aspect='equal', origin='upper')
    
    # Overlay boundary
    mainland.dissolve().boundary.plot(ax=ax, color='white', linewidth=0.5, alpha=0.7)
    
    ax.set_title(year, fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude')
    if i == 0:
        ax.set_ylabel('Latitude')
    else:
        ax.set_yticklabels([])

# Add colorbar
cbar = fig.colorbar(im, ax=axes, orientation='horizontal', fraction=0.04, pad=0.05,
                    shrink=0.8)
cbar.set_label('Radiance (nW/cm$^2$/sr)', fontsize=11)

plt.suptitle('VIIRS VNP46A4 — Nighttime Radiance (Mainland Portugal)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'portugal_darksky_classification.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print(f"Saved: {VIZ_DIR / 'portugal_darksky_classification.png'}")
plt.show()

## 8. Inter-Annual Comparison

Positive = brightening (new development). Negative = dimming.

In [ ]:
# Compute year-over-year change
diff_2024_2023 = cleaned_arrays['2024'].values - cleaned_arrays['2023'].values
diff_2025_2024 = cleaned_arrays['2025'].values - cleaned_arrays['2024'].values
diff_total = cleaned_arrays['2025'].values - cleaned_arrays['2023'].values

print("Inter-annual radiance change (nW/cm2/sr):")
print(f"{'Period':<15} {'Mean':<10} {'Median':<10} {'Std':<10}")
print("-" * 45)
for label, diff in [('2024-2023', diff_2024_2023),
                    ('2025-2024', diff_2025_2024),
                    ('2025-2023', diff_total)]:
    valid = diff[~np.isnan(diff)]
    print(f"{label:<15} {valid.mean():<10.4f} {np.median(valid):<10.4f} {valid.std():<10.4f}")

In [ ]:
# Spatial map of total change (2025 - 2023)
fig, ax = plt.subplots(figsize=(8, 10))

ds_ref = datasets['2023']
x_coords = ds_ref.coords['x'].values
y_coords = ds_ref.coords['y'].values
extent = [x_coords.min(), x_coords.max(), y_coords.min(), y_coords.max()]

im = ax.imshow(diff_total, extent=extent, cmap='RdBu_r', vmin=-2, vmax=2,
               interpolation='nearest', aspect='equal', origin='upper')
mainland.dissolve().boundary.plot(ax=ax, color='black', linewidth=0.5)

cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.02)
cbar.set_label('Radiance change (nW/cm$^2$/sr)', fontsize=10)

ax.set_title('Radiance Change 2025 vs 2023\n(Red = brightening, Blue = dimming)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(VIZ_DIR / 'inter_annual_variability.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print(f"Saved: {VIZ_DIR / 'inter_annual_variability.png'}")
plt.show()

## 9. Cache Inventory

In [ ]:
# List all VIIRS cache files
print(f"{'File':<40} {'Size (MB)':<10} {'Created':<22} {'Region':<12}")
print("-" * 84)

viirs_files = sorted(CACHE_DIR.glob('VNP46A4_*.joblib'))
total_size = 0

for f in viirs_files:
    size_mb = f.stat().st_size / 1e6
    total_size += size_mb
    
    meta_file = f.with_suffix('.meta.json')
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        created = meta.get('created_at', 'unknown')[:19]
        region = meta.get('parameters', {}).get('region', 'unknown')
    else:
        created = 'no metadata'
        region = 'unknown'
    
    print(f"{f.name:<40} {size_mb:<10.1f} {created:<22} {region:<12}")

print(f"\nTotal VIIRS cache: {total_size:.1f} MB ({len(viirs_files)} files)")

## Summary

3 years of VNP46A4 loaded and verified. Next: temporal compositing and classification in `02_darksky_classification.ipynb`.